# Phase 9 — Exploratory Data Analysis (viewer)

**This notebook computes nothing.** Every number it shows comes from
`src/evaluation/profile.py` and `src/evaluation/sampling.py`, which are under
`pytest` and are re-run by `python scripts/run_eda.py`.

That split is deliberate. A notebook's outputs are stored state: they drift from the
code that produced them, nothing fails when they do, and a reviewer cannot tell a
stale cell from a fresh one. `PROJECT_PLAN.md` asks for notebook EDA, and this is it —
but the arithmetic behind a published figure has to be testable, so the notebook is a
**viewer over tested functions**, not a place where results are derived.

**Read `reports/eda.md` for the analysis.** This is for poking at the corpus
interactively.

---

Three things that will mislead you if you forget them:

1. **The corpus is 100% synthetic** (OPEN-011). Every distribution is a distribution
   over a template grammar this project wrote. None of it is evidence about athletes.
2. **The unit is the utterance**, not the record. 1,200 records became 4,141
   utterances. Phase 7's per-record numbers are not comparable with anything here.
3. **`generation_spec` is not a label.** It records what the generator planted. Use it
   to describe the template bank and never as ground truth.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.profile import load_interim, profile_corpus
from src.evaluation.sampling import build_plan

# load_interim is the ONLY loader. There is no path-taking variant, because a
# function that accepts an arbitrary directory is a function that can be pointed
# at data/raw/ — which is not de-identified and which Phase 9 must not read.
records = load_interim("synth_precomp_v1")
len(records)

## Shape and length

In [ ]:
profile = profile_corpus(records, "synth_precomp_v1")

for dist in (
    profile.utterance_chars,
    profile.utterance_tokens,
    profile.record_chars,
    profile.record_tokens,
    profile.utterances_per_record,
):
    print(dist.as_row())

## Vocabulary

Quote **MATTR**, not raw TTR. TTR falls mechanically as a corpus grows — its
denominator is unbounded and its numerator saturates — so the TTR over 4,141
utterances is not comparable with the TTR Phase 7 reported over 1,200 records even
though the text is identical. MATTR averages fixed-length windows, so the length
confound is gone by construction.

In [ ]:
v = profile.vocabulary_with_placeholders
print(f"types={v.types}  tokens={v.tokens}  raw TTR={v.ttr:.4f}  MATTR-50={v.mattr:.4f}")
print(f"hapax={v.hapax} ({v.hapax_rate:.1%} of types)")
print(f"placeholder tokens in corpus: {profile.placeholder_token_count}")
v.top_types[:15]

## Duplicates — the largest finding of this phase

73.6% of utterances share their exact text with another. The mechanism is
segmentation: the generator's discourse suffixes and neutral sentences are appended
whole, are rendered *without* the near-synonym variation layer, and Phase 8 then cuts
each into its own utterance. At record level the duplicate rate is 5.0%.

In [ ]:
u = profile.utterance_duplicates
print(f"utterances={u.n}  distinct={u.distinct} ({u.distinct_rate:.1%})")
print(f"exact duplicate rate {u.exact_duplicate_rate_ci}")
print(f"near duplicate rate  {u.near_duplicate_rate_ci} (Jaccard >= {u.jaccard_threshold})")
u.most_repeated[:8]

## Junk, coverage, strata

In [ ]:
print(profile.junk.flagged, profile.junk.by_check, profile.junk.flagged_rate_ci)
print()
for name, interval in profile.coverage.items():
    print(f"{name:<28} {interval}")
print()
for name, counts in profile.strata.items():
    print(name, counts)

## Generator metadata — NOT labels

Kept in a separately named type so it cannot be used by accident. Note the
replication: Phase 8 copies the record's `generation_spec` onto every utterance cut
from it, so the per-utterance counts are the per-record counts multiplied by ~3.45
and mean nothing on their own.

In [ ]:
gm = profile.generator_metadata
print(gm.warning)
print()
print("records planting each construct:", gm.planted_construct_records)
print("templates:", gm.template_count)
print("constructs per record:", gm.constructs_per_record)

## The gold-set sampling plan

See `reports/eda.md` §7 for the reasoning. The short version: drawing from the test
side of `template_disjoint_split` leaves `appraisal_orientation` with zero items, so
templates are partitioned **per construct** instead.

In [ ]:
plan = build_plan(records, "synth_precomp_v1")
print("disjoint:", plan.partition.is_disjoint)
print("constructs with no holdout:", plan.partition.constructs_without_holdout or "none")
print("leakage-safe:", plan.is_leakage_safe)
print()
print("gold_eval:", plan.gold_eval.size, "| gold_dev:", plan.gold_dev.size)
print("eligibility:", plan.gold_eval.eligibility)
print()
print("below the 40-positive floor:", plan.gold_eval.constructs_below_floor)